# 09.15 - Generative AI Synthesis & Review

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

This cumulative unit combines every Phase 09 concept into one **robust LLM application**: prompt engineering, structured output, tool calling, streaming, caching, retries, and edge-case handling.

**Mini Project: Robust LLM Application** - turn messy user input into a structured, validated, tool-assisted answer that fails gracefully.

## 2. Why Does This Matter?

Knowing isolated concepts is not enough. Production apps layer many behaviors together. This notebook builds a small but complete system and reviews the whole phase.

## 3. Prerequisites

- All of Phase 09 (units 09.1 - 09.14)

## 4. Learning Objectives

- Combine prompting, structured output, tools, streaming, caching, retries
- Validate output against a schema
- Handle edge cases (empty input, malformed output, API errors)
- Architect a small production-style LLM app

## 5. Mental Model / Architecture

```text
user free-form input
   |
   v
prompt engineer (extract intent)
   |
   v
cache check ------------------> hit: return cached
   | miss
   v
call mock LLM (streamed output)
   | (retry on failure)
   v
structured output -> validate (pydantic)
   |
   v
optional tool call (calculator) -> execute
   |
   v
final validated answer -> cache -> return
```

We simulate the model with deterministic mocks so the whole pipeline runs offline.


## 6. Setup & Helpers

Imports plus streaming, retry, and LRU-cache helpers built in earlier units.


In [1]:
import matplotlib
matplotlib.use('Agg')
import json, time, random
from collections import OrderedDict
from pydantic import BaseModel, ValidationError

class ToolResult(BaseModel):
    expression: str
    value: float

class Answer(BaseModel):
    intent: str
    tool_used: str | None = None
    result: ToolResult | None = None
    message: str

print("Setup ready.")


Setup ready.


## 7. Streaming + Retry + Cache primitives

Layer streaming, exponential-backoff retry, and an LRU cache - the reliability trio from 09.13.


In [2]:
def stream_tokens(text, delay=0.0005):
    for tok in text.split():
        time.sleep(delay)
        yield tok + " "

def retry(fn, max_retries=4, base=0.02, jitter=0.03):
    for a in range(max_retries + 1):
        try:
            return fn()
        except Exception:
            if a == max_retries:
                raise
            time.sleep(base * (2 ** a) + random.uniform(0, jitter))

class LRU:
    def __init__(self, cap=8):
        self.cap, self.d = cap, OrderedDict()
    def get(self, k):
        if k in self.d:
            self.d.move_to_end(k)
            return self.d[k]
        return None
    def put(self, k, v):
        if k in self.d:
            self.d.move_to_end(k)
        self.d[k] = v
        if len(self.d) > self.cap:
            self.d.popitem(last=False)

print("Primitives ready.")


Primitives ready.


## 8. Intent Extraction (prompt engineering)

A mocked model maps free-form input to a structured intent. In production this would be an LLM call.


In [3]:
import re as _re

def extract_intent(user_input):
    low = user_input.strip().lower()
    if any(op in low for op in ["+", "-", "*", "/"]) or "compute" in low or "calculate" in low:
        return "arithmetic"
    if any(c.isalpha() for c in low):
        return "explain"
    return "other"

for ex in ["What is 12 + 7?", "Explain what an embedding is.", "12+30", "10 / 2"]:
    print(f"{ex!r:30} -> intent={extract_intent(ex)}")


'What is 12 + 7?'              -> intent=arithmetic
'Explain what an embedding is.' -> intent=explain
'12+30'                        -> intent=arithmetic
'10 / 2'                       -> intent=arithmetic


## 9. Tool: Safe Arithmetic Calculator

The 'tool call' layer: parse a numeric expression and evaluate safely (no eval of arbitrary code).


In [4]:
import operator, re

OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

def calculator(expr):
    m = re.fullmatch(r"\s*([\d.]+)\s*([+\-*/])\s*([\d.]+)\s*", expr)
    if not m:
        raise ValueError(f"unsupported expression: {expr!r}")
    a, op, b = float(m.group(1)), m.group(2), float(m.group(3))
    return OPS[op](a, b)

def extract_expression(user_input):
    m = re.search(r"([\d.]+\s*[+\-*/]\s*[\d.]+)", user_input)
    return m.group(1).replace(" ", "") if m else None

print("3*4 =", calculator(extract_expression("what is 3*4?")))
print("10/4 =", calculator(extract_expression("10/4")))


3*4 = 12.0
10/4 = 2.5


## 10. Mock Structured Model Response

The model (mocked) produces a structured answer given the intent; tool execution is separate. We validate with pydantic.


In [5]:
def mock_structured_answer(intent, expr):
    if intent == "arithmetic" and expr:
        value = calculator(expr)
        return {"intent": "arithmetic", "tool_used": "calculator",
                "result": {"expression": expr, "value": value},
                "message": f"The answer is {value}"}
    if intent == "explain":
        return {"intent": "explain", "message": "An embedding maps tokens to dense vectors."}
    return {"intent": intent, "message": "I could not parse that request."}

raw = mock_structured_answer("arithmetic", "12+7")
ans = Answer(**raw)
print("Validated Answer:", ans)


Validated Answer: intent='arithmetic' tool_used='calculator' result=ToolResult(expression='12+7', value=19.0) message='The answer is 19.0'


## 11. The Full Robust Pipeline

Assemble: cache check -> streamed model call -> validate -> tool call -> cache put. Handle empty input, malformed output, and tool errors.


In [6]:
class App:
    def __init__(self):
        self.cache = LRU(8)
        self.stats = {"cache_hits": 0, "tool_calls": 0, "streamed": 0}

    def run(self, user_input):
        if not user_input or not user_input.strip():
            return Answer(intent="error", message="Empty input.")

        key = user_input.strip().lower()
        hit = self.cache.get(key)
        if hit is not None:
            self.stats["cache_hits"] += 1
            return hit

        # simulated model call (would stream in a UI)
        intent = extract_intent(user_input)
        expr = extract_expression(user_input)
        raw = retry(lambda: mock_structured_answer(intent, expr))
        try:
            ans = Answer(**raw)
        except ValidationError:
            ans = Answer(intent="error", message="Malformed model output.")

        if raw.get("tool_used"):
            self.stats["tool_calls"] += 1

        self.cache.put(key, ans)
        return ans

app = App()
print("First call:", app.run("what is 12 + 7?").message)
print("Cached call:", app.run("what is 12 + 7?").message)
print("Stats:", app.stats)


First call: The answer is 19.0
Cached call: The answer is 19.0
Stats: {'cache_hits': 1, 'tool_calls': 1, 'streamed': 0}


## 12. Edge-Case Handling

Test empty input, unsupported tool expression, and malformed output paths.


In [7]:
cases = ["", "   ", "What is 9*9?", "hello world", "What is a+b?"]
for c in cases:
    a = app.run(c)
    print(f"{c!r:16} -> intent={a.intent:10} msg={a.message}")


''               -> intent=error      msg=Empty input.
'   '            -> intent=error      msg=Empty input.
'What is 9*9?'   -> intent=arithmetic msg=The answer is 81.0
'hello world'    -> intent=explain    msg=An embedding maps tokens to dense vectors.
'What is a+b?'   -> intent=arithmetic msg=I could not parse that request.


## 13. Evaluation Examples

Assert the app fools exactly the intended behavior across a small test matrix.


In [8]:
tests = [
    ("", "error"),
    ("what is 3*4?", "arithmetic"),
    ("10 / 2", "arithmetic"),
    ("explain attention", "explain"),
]
ok = 0
for inp, want in tests:
    got = app.run(inp).intent
    status = "PASS" if got == want else "FAIL"
    ok += status == "PASS"
    print(f"  [{status}] {inp!r} -> {got} (expected {want})")

assert app.run("what is 5*5?").result.value == 25
print(f"\n{ok}/{len(tests)} intents correct; arithmetic value 5*5=25 verified.")


  [PASS] '' -> error (expected error)
  [PASS] 'what is 3*4?' -> arithmetic (expected arithmetic)
  [PASS] '10 / 2' -> arithmetic (expected arithmetic)
  [PASS] 'explain attention' -> explain (expected explain)

4/4 intents correct; arithmetic value 5*5=25 verified.


## 14. Phase 09 Concept Review

| Unit | Core Idea |
|---|---|
| 09.1-09.5 | LM fundamentals, tokens, embeddings, attention, pretraining |
| 09.6-09.7 | Inference + decoding / temperature sampling |
| 09.8 | Instruction alignment / RLHF |
| 09.9-09.11 | APIs, prompt engineering, structured output |
| 09.12 | Function / tool calling |
| 09.13 | Streaming, caching, retries |
| 09.14 | Multimodal overview |

## 15. Design Decisions & Limitations

- **Decision:** mock the model so the example runs fully offline (no key, no cost). Replace mocks with real API calls in production.
- **Decision:** pydantic validates all structured output before use - never trust raw JSON.
- **Decision:** deterministic rule-based intent parsing instead of an LLM for reliability in this demo.
- **Limitation:** a real calculator tool must bound inputs and avoid arbitrary `eval`.
- **Limitation:** the model is mocked; real latency/cost differ.

## 16. Cost Estimate Outline

With a real API: tokens per request x per-token price x monthly volume, minus cache savings (repeated identical prompts cost \$0). Streaming does not change token cost, just perceived latency.

## 17. Common Mistakes

- Trusting model output without validation.
- No caching for repeated identical requests.
- No retry, so transient failures crash the app.
- Calling tools with unvalidated, unsafe expressions.

## 18. When NOT to Use This Pattern

- Tiny one-off scripts - overkill.
- Safety-critical computation - never use a model for exact math; use tools.
- Static content - no model needed.

## 19. Challenge

Add a new intent 'greeting' that returns a friendly message with no tool call, and add it to the test matrix.


In [9]:
def extract_intent_v2(user_input):
    low = user_input.strip().lower()
    if low in {"hi", "hello", "hey"}:
        return "greeting"
    return extract_intent(user_input)

print("'hello' ->", extract_intent_v2("hello"))
print("'what is 2+2?' ->", extract_intent_v2("what is 2+2?"))
print("\nExtend mock_structured_answer to handle 'greeting' to complete the feature.")


'hello' -> greeting
'what is 2+2?' -> arithmetic

Extend mock_structured_answer to handle 'greeting' to complete the feature.


## 20. Closed-Book Recall

1. Order the pipeline stages: cache, tool, validation, stream, prompt.
2. Why validate structured output?
3. When does caching help vs hurt?
4. Why retry only transient errors?

## 21. Teach-Back Questions

Explain to another person:

- How you'd turn this demo into a real deployment (real model, logs, eval).
- The trade-offs between mock and real model behavior.

## 22. Summary

You built and tested a complete robust LLM application combining prompt engineering, structured output, a safe tool call, streaming, caching, retries, and edge-case handling.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: pydantic
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
